# Retrieve `ibm_fez` holonomy archive\n\nReconstructs `hardware/results/ibm_fez_holonomy_20260709.json` (cited by note v3) from IBM Quantum job `d986q62f47jc73a7hm2g`.\n\nNeeds `QISKIT_IBM_TOKEN` (and optionally `QISKIT_IBM_CRN`) in Colab **Secrets** (key icon, left sidebar). Retrieve soon - IBM job retention is limited.

In [ ]:
%pip install -q qiskit-ibm-runtime

In [ ]:
# CELL 2 - connect (token from Colab userdata, same pattern as the multibackend notebook)
import os, json, datetime
from google.colab import userdata
from qiskit_ibm_runtime import QiskitRuntimeService

TOKEN = userdata.get("QISKIT_IBM_TOKEN")
try:
    CRN = userdata.get("QISKIT_IBM_CRN")
except Exception:
    CRN = None
service = QiskitRuntimeService(token=TOKEN, instance=CRN) if CRN else QiskitRuntimeService(token=TOKEN)
print("connected")

In [ ]:
# CELL 3 - fetch the pinned job and archive raw counts
JOB_ID = "d986q62f47jc73a7hm2g"
OUT = "/content/ibm_fez_holonomy_20260709.json"

job = service.job(JOB_ID)
res = job.result()
payload = {
    "job_id": JOB_ID,
    "backend": job.backend().name if callable(getattr(job, "backend", None)) else None,
    "retrieved": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "pinned_figures": {
        "loop_phase_deg": [-2.93, 1.55],
        "pm_S": [4.6125, 0.0173],
        "nc_bound": 4,
    },
    "results": [],
}
for i, pub in enumerate(res):
    entry = {"pub": i, "metadata": dict(getattr(pub, "metadata", {}) or {})}
    data = pub.data
    for field in data:
        bits = getattr(data, field)
        try:
            entry.setdefault("counts", {})[field] = bits.get_counts()
        except Exception:
            entry.setdefault("raw", {})[field] = str(bits)
    payload["results"].append(entry)

with open(OUT, "w") as fh:
    json.dump(payload, fh, indent=1)
print(f"archived {len(payload['results'])} pubs -> {OUT}")

In [ ]:
# CELL 4 - quick sanity: show pub metadata and total shots per pub
for e in payload["results"]:
    tot = sum(sum(c.values()) for c in e.get("counts", {}).values()) if e.get("counts") else None
    print(f'pub {e["pub"]}: shots={tot} meta={ {k: e["metadata"][k] for k in list(e["metadata"])[:4]} }')
print()
print("check these against the pinned figures before committing:")
print("  loop phase -2.93 deg +/- 1.55 ; PM S = 4.6125 +/- 0.0173 (bound 4)")

In [ ]:
# CELL 5 - download; then place the file at hardware/results/ in the repo via GitHub Desktop
from google.colab import files
files.download(OUT)